In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

import os
import json
import pickle
import random

SEED = 42

# Fix random seeds so training, data splitting, and downstream analysis
# remain as reproducible as possible across runs.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# All training artifacts are stored here and later reused by analysis notebooks.
ARTIFACTS_DIR = Path("../../model_training/models_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# cVAE Training for AMP Generation

This notebook trains a conditional variational autoencoder (cVAE) for antimicrobial peptide generation.

Pipeline:
1. load and clean the preprocessed AMP dataset,
2. build a character-level vocabulary,
3. tokenize peptide sequences and construct condition vectors,
4. train a GRU-based cVAE,
5. save all artifacts required for reproducible downstream analysis.

## 1. Load and clean the dataset

We load the merged AMP dataset, keep only the columns required for modeling, filter sequences to canonical amino acids, and remove duplicate peptides by sequence.

This ensures that the training data is syntactically clean and that repeated records do not inflate reconstruction or generation quality.

In [2]:
# Try a small set of candidate paths so the notebook works both
# from the repository root and from the notebooks directory.
candidates = [
    Path("Data/processed/master_dataset.csv"),
    Path("../../Data/processed/master_dataset.csv"),
]

path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError("master_dataset.csv was not found in the expected data directories.")

df = pd.read_csv(path)

id_col = 'APD ID'
seq_col = 'Sequence'
len_col = 'Length'
condition_cols = [
    'is_anti_gram_positive', 'is_anti_gram_negative',
    'is_antifungal', 'is_antiviral', 'is_antiparasitic', 'is_anticancer'
]

existing_cols = [col for col in [id_col, seq_col, len_col] + condition_cols if col in df.columns]
df = df[existing_cols].dropna(subset=[seq_col])

# Restrict training data to the 20 canonical amino acids.
# This removes sequences with unusual or ambiguous symbols and keeps
# the character-level vocabulary biologically consistent.
canonical_aas = set("ACDEFGHIKLMNPQRSTVWY")

df[seq_col] = df[seq_col].astype(str).str.upper()
df = df[df[seq_col].map(lambda s: all(ch in canonical_aas for ch in s))]
# Deduplicate by sequence so the model does not see the same peptide
# multiple times through merged database sources.
df = df.drop_duplicates(subset=[seq_col]).reset_index(drop=True)

for col in condition_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

df["Length"] = df["Length"].astype("Int64")
print(f"Size after cleaning: {df.shape}")
df.head()

Size after cleaning: (6012, 9)


,APD ID,Sequence,Length,is_anti_gram_positive,is_anti_gram_negative,is_antifungal,is_antiviral,is_antiparasitic,is_anticancer
0,AP05452,AAAGKHKNKGKKNGKHNGWK,20,1,1,1,0,0,0
1,AP05433,AAAHKHGHGHGKHKNKGKKN,20,1,1,1,0,0,0
2,AP01515,AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,40,1,1,0,0,0,0
3,AP00612,AAEFPDFYDSEEQMGPHQEAEDEKDRADQRVLTEEEKKELENLAAM...,60,1,1,0,0,0,0
4,AP01914,AAFRGCWTKNYSPKPCL,17,1,0,0,0,0,0


## 2. Build the tokenizer and prepare model inputs

Peptides are modeled at the character level. We build a vocabulary over amino-acid symbols and add special tokens:
`<PAD>`, `<SOS>`, `<EOS>`, `<UNK>`.

Sequences are then tokenized to a fixed maximum length so they can be batched efficiently.

In [3]:
all_chars = set()
for seq in df[seq_col]:
    all_chars.update(seq)

special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
vocab_list = special_tokens + sorted(all_chars)
char2idx = {ch: i for i, ch in enumerate(vocab_list)}
idx2char = {i: ch for ch, i in char2idx.items()}
vocab_size = len(vocab_list)

print(f"Vocab size: {vocab_size}")
print("Words:", list(all_chars)[:10])

Vocab size: 24
Words: ['H', 'K', 'F', 'T', 'P', 'V', 'R', 'Y', 'C', 'N']


In [4]:
def tokenize(seq, char2idx, max_len, add_sos_eos=True):
    """
    Convert a peptide string into a fixed-length list of token ids.

    We optionally add <SOS> and <EOS> so the decoder learns explicit
    sequence start and stop boundaries. Shorter sequences are padded
    and longer ones are truncated to max_len.
    """
    tokens = [char2idx.get(ch, char2idx['<UNK>']) for ch in seq]
    if add_sos_eos:
        tokens = [char2idx['<SOS>']] + tokens + [char2idx['<EOS>']]
    if len(tokens) < max_len:
        tokens += [char2idx['<PAD>']] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    return tokens

# Reserve two extra positions for <SOS> and <EOS>.
max_len = df[len_col].max() + 2

tokenized_seqs = []
for seq in df[seq_col]:
    tokenized = tokenize(seq, char2idx, max_len, add_sos_eos=True)
    tokenized_seqs.append(tokenized)

conditions = df[condition_cols].values.astype(np.float32)

real_lengths = np.array([len(seq) + 2 for seq in df[seq_col]])
real_lengths = np.minimum(real_lengths, max_len)

print(f"Max len after pad: {real_lengths}")

Max len after pad: [22 22 42 ... 25 28 22]


## 3. Create dataset objects and deterministic data splits

We construct a PyTorch dataset containing tokenized sequences, sequence lengths, and condition vectors.

The train/validation/test split is deterministic and its indices are saved to disk so that post-training analysis can reproduce the exact same partition.

In [5]:
class AMPDataset(Dataset):
    """Dataset of tokenized peptide sequences, effective lengths, and condition vectors."""
    def __init__(self, sequences, lengths, conditions):
        self.sequences = torch.LongTensor(sequences)
        self.lengths = torch.LongTensor(lengths)
        self.conditions = torch.FloatTensor(conditions)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.lengths[idx], self.conditions[idx]

dataset = AMPDataset(tokenized_seqs, real_lengths, conditions)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
# Save the exact split indices so evaluation later uses the same examples
# rather than a newly sampled partition.
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

with open(ARTIFACTS_DIR / "split_indices.pkl", "wb") as f:
    pickle.dump({
        "train_idx": train_dataset.indices,
        "val_idx": val_dataset.indices,
        "test_idx": test_dataset.indices
    }, f)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 4809, Val: 601, Test: 602


## 4. Define the GRU-based cVAE

The encoder maps a peptide sequence and its condition vector to the latent posterior parameters `(mu, logvar)`.
The decoder reconstructs the sequence autoregressively from the sampled latent vector `z` and the same condition vector.

Condition information is injected both into the encoder and into the decoder.

In [6]:
# cVAE architecture

class Encoder(nn.Module):
    """
    Encode a peptide sequence into the parameters of the latent posterior.

    The final GRU hidden state is concatenated with the condition vector,
    allowing the latent distribution q(z | x, c) to depend on both sequence
    content and the requested biological condition.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=char2idx['<PAD>'])
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=False)
        self.fc_mu = nn.Linear(hidden_dim + cond_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim + cond_dim, latent_dim)

    def forward(self, x, lengths, cond):
        # x: (batch, seq_len)
        # print("x.shape:", x.shape)
        # print("lengths.max():", int(lengths.max()), "x.size(1):", x.size(1))
        # assert int(lengths.max()) <= x.size(1)
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)  # hidden: (1, batch, hidden_dim)
        hidden = hidden.squeeze(0)    # (batch, hidden_dim)
        hidden_cond = torch.cat([hidden, cond], dim=1)  # (batch, hidden_dim+cond_dim)
        mu = self.fc_mu(hidden_cond)
        logvar = self.fc_logvar(hidden_cond)
        return mu, logvar

class Decoder(nn.Module):
    """
    Autoregressive decoder conditioned on z and c ONLY through the initial hidden state.
    Removing z and c from every step forces the decoder to rely on the latent code.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=char2idx['<PAD>'])
        self.input_dropout = nn.Dropout(dropout)
        # GRU получает только токен, без z и cond
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        # Начальное скрытое состояние формируется из z и cond
        self.init_h = nn.Linear(latent_dim + cond_dim, hidden_dim)

    def forward(self, x, lengths, z, cond):
        # x: (batch, seq_len)
        z_cond = torch.cat([z, cond], dim=1)           # (B, latent+cond)
        h0 = self.init_h(z_cond).unsqueeze(0)          # (1, B, hidden_dim)

        embedded = self.embedding(x)                   # (B, T, embed_dim)
        embedded = self.input_dropout(embedded)

        outputs, _ = self.gru(embedded, h0)            # (B, T, hidden_dim)
        logits = self.fc_out(outputs)                  # (B, T, vocab)
        return logits

class CVAE(nn.Module):
    """Conditional VAE for character-level peptide generation."""
    def __init__(self, vocab_size, embed_dim, hidden_dim_enc, hidden_dim_dec, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim_enc, latent_dim, cond_dim, dropout)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim_dec, latent_dim, cond_dim, dropout)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, src, lengths, cond):
        mu, logvar = self.encoder(src, lengths, cond)
        z = self.reparameterize(mu, logvar)
        # Teacher forcing: predict the next token given all previous ground-truth tokens.
        decoder_input = src[:, :-1]
        target = src[:, 1:]
        logits = self.decoder(decoder_input, lengths - 1, z, cond)
        return logits, target, mu, logvar

## 5. Train the model

The model is optimized using reconstruction loss plus KL regularization.

We use:
- KL warmup during early epochs,
- gradient clipping for training stability,
- early stopping based on validation loss.

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

embed_dim = 64
hidden_dim_enc = 128
hidden_dim_dec = 64
latent_dim = 32
cond_dim = len(condition_cols)
dropout = 0.2
lr = 1e-3

model = CVAE(vocab_size, embed_dim, hidden_dim_enc, hidden_dim_dec, latent_dim, cond_dim, dropout).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss(ignore_index=char2idx['<PAD>'])

def loss_function(logits, target, mu, logvar, beta=1.0):
    """
    Compute the cVAE objective = reconstruction loss + beta * KL.

    Reconstruction is token-level cross-entropy with PAD ignored.
    KL regularizes the latent space toward a standard normal prior.
    """
    # logits: (batch, seq_len, vocab)
    # target: (batch, seq_len)
    recon_loss = criterion(logits.reshape(-1, vocab_size), target.reshape(-1))
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    # Average KL over the batch so it stays on a comparable scale across batch sizes.
    kl_loss /= target.size(0)
    kl_loss = torch.clamp(kl_loss, min=0.3)
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

Using device: cpu


In [8]:
import os
output_path = "../../model_training/models_artifacts"
def train_epoch(model, loader, optimizer, beta=1.0):
    """Run one training epoch."""
    model.train()
    total_loss = 0
    for src, lengths, cond in loader:
        src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
        optimizer.zero_grad()
        logits, target, mu, logvar = model(src, lengths, cond)
        loss, recon, kl = loss_function(logits, target, mu, logvar, beta)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, beta=1.0):
    """Run one evaluation epoch without gradient updates."""
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, lengths, cond in loader:
            src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
            logits, target, mu, logvar = model(src, lengths, cond)
            loss, _, _ = loss_function(logits, target, mu, logvar, beta)
            total_loss += loss.item()
    return total_loss / len(loader)

num_epochs = 100
best_val_loss = float('inf')
patience = 8
counter = 0

train_losses, val_losses = [], []
train_recon_losses, val_recon_losses = [], []
train_kl_losses, val_kl_losses = [], []

for epoch in range(num_epochs):
    # KL warmup reduces posterior collapse risk at the start of training.
    beta = 1 / (1 + np.exp(-0.15 * (epoch - 40)))

    model.train()
    total_train, total_recon, total_kl = 0, 0, 0
    for src, lengths, cond in train_loader:
        src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
        optimizer.zero_grad()

        logits, target, mu, logvar = model(src, lengths, cond)
        loss, recon, kl = loss_function(logits, target, mu, logvar, beta=beta)

        loss.backward()
        # Gradient clipping improves stability for autoregressive recurrent training.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()

    train_loss = total_train / len(train_loader)
    train_recon = total_recon / len(train_loader)
    train_kl = total_kl / len(train_loader)

    model.eval()
    total_val, total_val_recon, total_val_kl = 0, 0, 0
    with torch.no_grad():
        for src, lengths, cond in val_loader:
            src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
            logits, target, mu, logvar = model(src, lengths, cond)
            loss, recon, kl = loss_function(logits, target, mu, logvar, beta=beta)

            total_val += loss.item()
            total_val_recon += recon.item()
            total_val_kl += kl.item()

    val_loss = total_val / len(val_loader)
    val_recon = total_val_recon / len(val_loader)
    val_kl = total_val_kl / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_recon_losses.append(train_recon)
    val_recon_losses.append(val_recon)
    train_kl_losses.append(train_kl)
    val_kl_losses.append(val_kl)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"beta={beta:.2f} | "
        f"train={train_loss:.4f} (recon={train_recon:.4f}, kl={train_kl:.4f}) | "
        f"val={val_loss:.4f} (recon={val_recon:.4f}, kl={val_kl:.4f})"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), ARTIFACTS_DIR / "best_cvae.pt")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(ARTIFACTS_DIR / "best_cvae.pt", map_location=device))

Epoch 1/100 | beta=0.00 | train=3.0499 (recon=3.0472, kl=1.1149) | val=2.8971 (recon=2.8922, kl=1.9995)
Epoch 2/100 | beta=0.00 | train=2.8703 (recon=2.8647, kl=1.9397) | val=2.8126 (recon=2.8045, kl=2.8300)
Epoch 3/100 | beta=0.00 | train=2.7983 (recon=2.7846, kl=4.1039) | val=2.7487 (recon=2.7303, kl=5.5180)
Epoch 4/100 | beta=0.00 | train=2.7409 (recon=2.7216, kl=4.9980) | val=2.7028 (recon=2.6806, kl=5.7411)
Epoch 5/100 | beta=0.00 | train=2.7021 (recon=2.6760, kl=5.7966) | val=2.6721 (recon=2.6418, kl=6.7359)
Epoch 6/100 | beta=0.01 | train=2.6731 (recon=2.6413, kl=6.0918) | val=2.6483 (recon=2.6170, kl=5.9965)
Epoch 7/100 | beta=0.01 | train=2.6508 (recon=2.6142, kl=6.0384) | val=2.6308 (recon=2.5926, kl=6.2978)
Epoch 8/100 | beta=0.01 | train=2.6314 (recon=2.5885, kl=6.1019) | val=2.6105 (recon=2.5673, kl=6.1453)
Epoch 9/100 | beta=0.01 | train=2.6158 (recon=2.5660, kl=6.1078) | val=2.5961 (recon=2.5443, kl=6.3509)
Epoch 10/100 | beta=0.01 | train=2.6044 (recon=2.5479, kl=5.9638

<All keys matched successfully>

## 6. Conditional sampling sanity check

After training, we generate a few example peptides under a selected condition to verify that the model can decode valid AMP-like sequences from the latent space.

In [9]:
def generate(model, cond, max_gen_len=None, temperature=1.0):
    """
    Sample a new peptide from the conditional prior.

    Temperature controls the sharpness of the token distribution:
    lower values make decoding more conservative, higher values increase diversity.
    """
    if max_gen_len is None:
        max_gen_len = max_len
    model.eval()
    with torch.no_grad():
        cond = torch.FloatTensor(cond).unsqueeze(0).to(device)
        z = torch.randn(1, latent_dim).to(device)
        z_cond = torch.cat([z, cond], dim=1)  # (1, latent+cond)
        h = model.decoder.init_h(z_cond).unsqueeze(0)  # (1, 1, hidden_dim)

        input_token = torch.LongTensor([[char2idx['<SOS>']]]).to(device)
        generated = []

        for _ in range(max_gen_len):
            embedded = model.decoder.embedding(input_token)   # (1, 1, embed_dim)
            output, h = model.decoder.gru(embedded, h)       # (1, 1, hidden_dim)
            logits = model.decoder.fc_out(output.squeeze(1)) # (1, vocab)
            probs = torch.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, 1).item()

            if next_token == char2idx['<EOS>']:
                break
            generated.append(next_token)
            input_token = torch.LongTensor([[next_token]]).to(device)

        seq = ''.join(idx2char[idx] for idx in generated
                      if idx not in [char2idx['<PAD>'], char2idx['<SOS>'], char2idx['<EOS>']])
        return seq

# Example condition vector: Gram-positive activity only.
cond_gram_pos = [0, 1, 0, 0, 0, 0]
for i in range(5):
    print(generate(model, cond_gram_pos, temperature=0.9))

CARGCSCFCKIRAPIRQGIPWLRVYCTACLR
MWRKLAKHSGHHIHGKIAGIAQGAALNLVSQF
KKIVKKIPKIFQ
KPPFQKIAKLLDKL
RRIKRWVRKVGKRIRRRY


## 7. Save artifacts for reproducibility

We save the vocabulary, configuration, split indices, and training history so that analysis notebooks can reconstruct the exact model setup without retraining.

In [10]:
# Save everything needed to reproduce analysis without retraining:
# tokenizer, model configuration, and per-epoch training curves.
with open(ARTIFACTS_DIR / "vocab.pkl", "wb") as f:
    pickle.dump({"char2idx": char2idx, "idx2char": idx2char, "max_len": max_len}, f)

with open(ARTIFACTS_DIR / "cvae_config.json", "w") as f:
    json.dump({
        "embed_dim": embed_dim,
        "hidden_dim_enc": hidden_dim_enc,
        "hidden_dim_dec": hidden_dim_dec,
        "latent_dim": latent_dim,
        "cond_dim": cond_dim,
        "dropout": dropout,
        "lr": lr,
        "seed": SEED,
        "condition_cols": condition_cols,
        "seq_col": seq_col,
        "len_col": len_col
    }, f, indent=2)

with open(ARTIFACTS_DIR / "training_history.pkl", "wb") as f:
    pickle.dump({
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_recon_losses": train_recon_losses,
        "val_recon_losses": val_recon_losses,
        "train_kl_losses": train_kl_losses,
        "val_kl_losses": val_kl_losses
    }, f)